<a href="https://colab.research.google.com/github/shpark030305-lgtm/OLMO_Fine-Tunning/blob/main/olmo2_lora_sft_dpo_workplace_english_colab_FIXED_comparison4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# OLMo2-1B QLoRA SFT + DPO Fine-Tuning Notebook

.

Flow:

1. Load the English CSV / JSONL / README files  
2. Convert the data into SFT text format  
3. Create DPO preference pairs: `prompt`, `chosen`, `rejected`  
4. Load OLMo2-1B Base in 4-bit  
5. Train only LoRA adapter parameters with SFT  
6. Save the SFT adapter  
7. Continue with DPO based on the SFT adapter  
8. Compare Base / SFT / SFT+DPO outputs  

This is **partial-parameter fine-tuning** using QLoRA/LoRA, not full fine-tuning of all model weights.


In [ ]:
# ============================================================
# 0. Install packages
# ============================================================
# pandas is pinned to the Colab-compatible version to avoid pandas 3.x conflicts.

!pip install -q -U transformers peft trl accelerate datasets bitsandbytes huggingface_hub matplotlib
!pip install -q "pandas==2.2.2"


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.3/80.3 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 77.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 842.4/842.4 kB 48.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 27.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 765.1/765.1 kB 37.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 113.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 14.6 MB/s eta 0:00:00


In [ ]:
import os
import gc
import json
import inspect
import random

# Force reinstall pyarrow and datasets to resolve potential binary incompatibility.
# This is a common workaround for "pyarrow.lib.IpcReadOptions size changed" errors in Colab.
# It should ideally be in the installation cell, but placed here due to specific cell modification request.
!pip install -q --force-reinstall pyarrow datasets

# Put the HF datasets cache in a stable Colab path instead of /tmp.
# This prevents pyarrow cache-file errors during TRL/DPO preprocessing.
os.environ["HF_DATASETS_CACHE"] = "/content/hf_datasets_cache"
os.environ["HF_HOME"] = "/content/hf_home"
os.environ["WANDB_DISABLED"] = "true"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.makedirs(os.environ["HF_DATASETS_CACHE"], exist_ok=True)
os.makedirs(os.environ["HF_HOME"], exist_ok=True)

import torch
import pandas as pd
import matplotlib.pyplot as plt

import datasets
from datasets import Dataset
datasets.disable_caching()

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import (
    LoraConfig,
    get_peft_model,
    TaskType,
    PeftModel,
    prepare_model_for_kbit_training,
)
from trl import SFTTrainer, SFTConfig, DPOTrainer, DPOConfig

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

BASE_ID = "allenai/OLMo-2-0425-1B"

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

def clear_gpu():
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()

def set_pad_token(tokenizer):
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    return tokenizer

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 3.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 3.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.7/41.7 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.2/94.2 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 120.0/120.0 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.4/203.4 kB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.5/73.5 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.8/78.8 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 150.3/150.3 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.2/100.2 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8

ValueError: pyarrow.lib.IpcReadOptions size changed, may indicate binary incompatibility. Expected 112 from C header, got 104 from PyObject

## 2. Upload the English dataset files

Upload these three files to Colab:

- `developmental_disability_work_sft_500_en.csv`
- `developmental_disability_work_sft_500_en.jsonl`
- `developmental_disability_work_sft_500_en_README.json`

The code checks both `/content` and `/mnt/data`.


In [ ]:
# ============================================================
# 2. Locate English files
# ============================================================

CANDIDATES = {
    "csv": [
        "/content/developmental_disability_work_sft_500_en.csv",
        "/mnt/data/developmental_disability_work_sft_500_en.csv",
    ],
    "jsonl": [
        "/content/developmental_disability_work_sft_500_en.jsonl",
        "/mnt/data/developmental_disability_work_sft_500_en.jsonl",
    ],
    "readme": [
        "/content/developmental_disability_work_sft_500_en_README.json",
        "/mnt/data/developmental_disability_work_sft_500_en_README.json",
    ],
}

def find_file(paths, label):
    for p in paths:
        if os.path.exists(p):
            print(f"{label} file found:", p)
            return p
    raise FileNotFoundError(f"{label} file was not found. Please upload it to Colab.")

CSV_PATH = find_file(CANDIDATES["csv"], "CSV")
JSONL_PATH = find_file(CANDIDATES["jsonl"], "JSONL")
README_PATH = find_file(CANDIDATES["readme"], "README")


In [ ]:
# ============================================================
# 3. Load and validate the English SFT dataset
# ============================================================

df = pd.read_csv(CSV_PATH).fillna("")
with open(README_PATH, "r", encoding="utf-8") as f:
    meta = json.load(f)

print("README:")
print(json.dumps(meta, ensure_ascii=False, indent=2)[:1600])

required_cols = ["id", "instruction", "input", "output", "task", "language", "source"]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

df = df[
    (df["instruction"].astype(str).str.strip() != "") &
    (df["output"].astype(str).str.strip() != "")
].copy()

print("Valid rows:", len(df))
print("\nTask distribution:")
print(df["task"].value_counts())
print("\nSource distribution:")
print(df["source"].value_counts())

display(df.head(5))


In [ ]:
# ============================================================
# 4. Build SFT text dataset
# ============================================================

def build_prompt_from_row(row):
    instruction = str(row["instruction"]).strip()
    input_text = str(row.get("input", "")).strip()

    prompt = f"### Instruction:\n{instruction}\n\n"
    if input_text:
        prompt += f"### Input:\n{input_text}\n\n"
    prompt += "### Response:\n"
    return prompt

def build_sft_text(row):
    return build_prompt_from_row(row) + str(row["output"]).strip()

sft_df = df.copy()
sft_df["text"] = sft_df.apply(build_sft_text, axis=1)

sft_dataset = Dataset.from_pandas(
    sft_df[["text", "task", "language", "source"]].sample(frac=1, random_state=SEED),
    preserve_index=False
)

split = sft_dataset.train_test_split(test_size=0.1, seed=SEED)
sft_train = split["train"]
sft_eval = split["test"]

print(sft_train)
print(sft_eval)
print("\nSFT sample:")
print(sft_train[0]["text"][:1200])


In [ ]:
# ============================================================
# 5. Build DPO preference pairs
# chosen = good answer
# rejected = intentionally bad answer
# ============================================================

def make_rejected(row):
    task = str(row["task"])

    if task == "work_support_situation_response":
        return "Just tell the worker to try harder. If the same mistake happens again, scolding the worker is the best way to stop it."
    if task == "easy_read_rewriting":
        return "The worker should just read the original sentence. If it is hard to understand, they should memorize it as it is."
    if task == "job_coaching_checklist":
        return "No checklist is needed. The worker should figure it out alone and finish as quickly as possible."
    if task == "manager_response_guidance":
        return "If the worker repeats a mistake, point it out publicly and make the worker take responsibility in front of others."
    if task == "safety_risk_situation":
        return "Just be careful. If there is not enough time, it is okay to skip safety checks."
    if task == "labor_rights_basics":
        return "Even if there is a problem, it is better to stay quiet and endure it so that the situation does not become bigger."
    if task == "customer_service_work_manners":
        return "If a customer gets angry, speak loudly back to the customer. There is no need to ask the person in charge for help."

    return "Handle the situation roughly without asking for help."

dpo_df = pd.DataFrame({
    "prompt": df.apply(build_prompt_from_row, axis=1),
    "chosen": df["output"].astype(str),
    "rejected": df.apply(make_rejected, axis=1),
    "task": df["task"].astype(str),
    "language": df["language"].astype(str),
    "source": df["source"].astype(str),
})

dpo_df = dpo_df.sample(frac=1, random_state=SEED).reset_index(drop=True)

DPO_JSONL_PATH = "/content/developmental_disability_work_dpo_pairs_500_en.jsonl"
DPO_CSV_PATH = "/content/developmental_disability_work_dpo_pairs_500_en.csv"

dpo_df.to_csv(DPO_CSV_PATH, index=False, encoding="utf-8-sig")
with open(DPO_JSONL_PATH, "w", encoding="utf-8") as f:
    for _, row in dpo_df.iterrows():
        f.write(json.dumps(row.to_dict(), ensure_ascii=False) + "\n")

# Use Dataset.from_list instead of Dataset.from_pandas.
# This keeps the dataset simple and avoids fragile temporary Arrow cache references.
dpo_records = dpo_df.to_dict("records")
dpo_dataset = Dataset.from_list(dpo_records)
dpo_split = dpo_dataset.train_test_split(test_size=0.1, seed=SEED)
dpo_train = dpo_split["train"]
dpo_eval = dpo_split["test"]

print(dpo_train)
print(dpo_eval)
print("\nDPO sample:")
print("PROMPT:", dpo_train[0]["prompt"][:500])
print("CHOSEN:", dpo_train[0]["chosen"][:300])
print("REJECTED:", dpo_train[0]["rejected"][:300])


In [ ]:
# ============================================================
# 6. Common generation function
# ============================================================

def generate_response(model, tokenizer, instruction, input_text="", max_new_tokens=180):
    model.eval()

    prompt = f"### Instruction:\n{instruction}\n\n"
    if input_text:
        prompt += f"### Input:\n{input_text}\n\n"
    prompt += "### Response:\n"

    device = next(model.parameters()).device
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    input_len = inputs["input_ids"].shape[1]

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    return tokenizer.decode(out[0][input_len:], skip_special_tokens=True).strip()


In [ ]:
# ============================================================
# 7. Base model response check
# ============================================================

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float32, # Changed to float32 to avoid BFloat16 issues
    bnb_4bit_use_double_quant=True,
)

base_tok = AutoTokenizer.from_pretrained(BASE_ID)
base_tok = set_pad_token(base_tok)
base_tok.padding_side = "right"

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_ID,
    quantization_config=bnb_config,
    torch_dtype=torch.float32, # Changed to float32 to avoid BFloat16 issues
    device_map="auto",
)
base_model.eval()

TEST_ITEMS = [
    {
        "instruction": "A worker with a developmental disability often forgets the work sequence. How should a support person help?",
        "input": "Situation: The worker is doing a cafe store cleanup task and often mixes up the order of organizing cups and wiping tables.",
    },
    {
        "instruction": "Rewrite the following sentence in plain language that a worker with a developmental disability can easily understand.",
        "input": "Workers must follow the scheduled working hours and break times.",
    },
    {
        "instruction": "Explain the safety rule for a wet floor situation in simple language for a worker with a developmental disability.",
        "input": "Risk situation: wet floor",
    },
]

base_outputs = []
for item in TEST_ITEMS:
    resp = generate_response(base_model, base_tok, item["instruction"], item["input"])
    base_outputs.append(resp)
    print("\n[BASE]")
    print("Q:", item["instruction"])
    print("A:", resp[:500])

del base_model
del base_tok
clear_gpu()

In [ ]:
# ============================================================
# 8. OLMo2-1B QLoRA SFT
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(BASE_ID)
tokenizer = set_pad_token(tokenizer)
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    BASE_ID,
    quantization_config=bnb_config,
    torch_dtype=torch.float32, # Changed to float32 to avoid BFloat16 issues
    device_map="auto",
)

model.config.use_cache = False
# model = prepare_model_for_kbit_training(model) # prepare_model_for_kbit_training already calls gradient_checkpointing_enable implicitly if use_gradient_checkpointing=True

# Removed explicit gradient_checkpointing_enable() call to avoid potential conflicts
# with persistent BFloat16 NotImplementedError in AMP gradient scaling.
# try:
#     model.gradient_checkpointing_enable()
# except Exception as e:
#     print("gradient_checkpointing_enable skipped:", repr(e))
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=False) # Explicitly disable gradient checkpointing in prepare_model_for_kbit_training

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
)

model = get_peft_model(model, lora_config)

# T4 BF16/AMP stability: keep trainable LoRA parameters in float32.
for name, param in model.named_parameters():
    if param.requires_grad:
        param.data = param.data.float()

model.print_trainable_parameters()

In [ ]:
# ============================================================
# 9. Version-compatible SFTConfig / SFTTrainer
# ============================================================

def make_sft_config():
    sig = inspect.signature(SFTConfig.__init__)
    params = sig.parameters

    kwargs = {
        "output_dir": "./olmo2_work_sft_adapter_en",
        "max_steps": 120,
        "per_device_train_batch_size": 1,
        "gradient_accumulation_steps": 8,
        "learning_rate": 2e-4,
        "lr_scheduler_type": "cosine",
        "warmup_steps": 10,

        # T4-safe setting: no AMP. The model is still loaded in 4-bit;
        # only LoRA parameters are trained.
        "fp16": False,
        "bf16": False,

        "logging_steps": 5,
        "save_steps": 60,
        "eval_steps": 60,
        "max_grad_norm": 0.0,
        "report_to": "none",
        "gradient_checkpointing": False,
    }

    if "eval_strategy" in params:
        kwargs["eval_strategy"] = "steps"
    elif "evaluation_strategy" in params:
        kwargs["evaluation_strategy"] = "steps"

    if "max_length" in params:
        kwargs["max_length"] = 512
    elif "max_seq_length" in params:
        kwargs["max_seq_length"] = 512

    if "dataset_text_field" in params:
        kwargs["dataset_text_field"] = "text"

    if "packing" in params:
        kwargs["packing"] = False

    clean_kwargs = {k: v for k, v in kwargs.items() if k in params}
    return SFTConfig(**clean_kwargs)

def make_sft_trainer(model, tokenizer, train_ds, eval_ds, args):
    sig = inspect.signature(SFTTrainer.__init__)
    params = sig.parameters

    kwargs = {
        "model": model,
        "train_dataset": train_ds,
        "eval_dataset": eval_ds,
        "args": args,
    }

    if "processing_class" in params:
        kwargs["processing_class"] = tokenizer
    elif "tokenizer" in params:
        kwargs["tokenizer"] = tokenizer

    return SFTTrainer(**kwargs)

sft_args = make_sft_config()
sft_trainer = make_sft_trainer(model, tokenizer, sft_train, sft_eval, sft_args)

print(sft_args)


In [ ]:
# ============================================================
# 10. Run SFT training
# ============================================================

print("Starting SFT training")
print("Accelerator mixed_precision:", getattr(sft_trainer.accelerator, "mixed_precision", "unknown"))

sft_result = sft_trainer.train()
print(sft_result)

sft_trainer.save_model("./olmo2_work_sft_adapter_en")
tokenizer.save_pretrained("./olmo2_work_sft_adapter_en")

print("SFT adapter saved: ./olmo2_work_sft_adapter_en")


In [ ]:
# ============================================================
# 11. Check SFT model responses
# ============================================================

sft_outputs = []
for item in TEST_ITEMS:
    resp = generate_response(model, tokenizer, item["instruction"], item["input"])
    sft_outputs.append(resp)
    print("\n[SFT]")
    print("Q:", item["instruction"])
    print("A:", resp[:500])

pd.DataFrame(sft_trainer.state.log_history).to_csv(
    "/content/sft_train_log_en.csv", index=False, encoding="utf-8-sig"
)

del sft_trainer
del model
clear_gpu()


In [ ]:
# ============================================================
# 12. Prepare DPO from the SFT adapter
# ============================================================

dpo_tokenizer = AutoTokenizer.from_pretrained(BASE_ID)
dpo_tokenizer = set_pad_token(dpo_tokenizer)
dpo_tokenizer.padding_side = "right"

dpo_base = AutoModelForCausalLM.from_pretrained(
    BASE_ID,
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    device_map="auto",
)

dpo_base.config.use_cache = False
dpo_base = prepare_model_for_kbit_training(dpo_base)

dpo_model = PeftModel.from_pretrained(
    dpo_base,
    "./olmo2_work_sft_adapter_en",
    is_trainable=True,
)

for name, param in dpo_model.named_parameters():
    if param.requires_grad:
        param.data = param.data.float()

dpo_model.print_trainable_parameters()


## DPO 오류 수정 포인트


이번 수정본
- `precompute_ref_log_probs=False`
- Hugging Face datasets cache를 `/content/hf_datasets_cache`로 고정
- `datasets.disable_caching()` 적용
- DPO dataset을 `Dataset.from_list()`로 생성
- SFT/DPO 모두 `fp16=False`, `bf16=False`로 T4 AMP/BF16 문제 회피

이미 성공한 SFT 흐름은 유지하고, DPOTrainer 생성 및 DPO 학습 단계만 더 안정적으로 바꿈


In [ ]:
# ============================================================
# 13. Version-compatible DPOConfig / DPOTrainer
# ============================================================
# Important fix:
# The previous notebook failed while precomputing/caching reference log probabilities:
# FileNotFoundError: /tmp/hf_datasets-.../cache-....arrow
#
# So this version disables `precompute_ref_log_probs` and computes reference
# log probabilities during training instead of writing temporary Arrow cache files.

def make_dpo_config():
    sig = inspect.signature(DPOConfig.__init__)
    params = sig.parameters

    kwargs = {
        "output_dir": "./olmo2_work_sft_dpo_adapter_en",
        "max_steps": 60,
        "per_device_train_batch_size": 1,
        "gradient_accumulation_steps": 8,
        "learning_rate": 5e-6,
        "lr_scheduler_type": "cosine",
        "warmup_steps": 5,

        # T4-safe setting: no AMP.
        "fp16": False,
        "bf16": False,

        "logging_steps": 5,
        "save_steps": 30,
        "eval_steps": 30,
        "beta": 0.1,
        "max_grad_norm": 0.0,
        "remove_unused_columns": False,
        "report_to": "none",
        "gradient_checkpointing": False,
    }

    if "eval_strategy" in params:
        kwargs["eval_strategy"] = "steps"
    elif "evaluation_strategy" in params:
        kwargs["evaluation_strategy"] = "steps"

    if "max_length" in params:
        kwargs["max_length"] = 512
    if "max_prompt_length" in params:
        kwargs["max_prompt_length"] = 256

    # 핵심 수정: True로 두면 TRL이 reference logprob를 임시 Arrow 파일로 캐싱하다가
    # Colab /tmp 파일이 사라져 FileNotFoundError가 발생할 수 있음.
    if "precompute_ref_log_probs" in params:
        kwargs["precompute_ref_log_probs"] = False

    clean_kwargs = {k: v for k, v in kwargs.items() if k in params}
    return DPOConfig(**clean_kwargs)

def make_dpo_trainer(model, tokenizer, train_ds, eval_ds, args):
    sig = inspect.signature(DPOTrainer.__init__)
    params = sig.parameters

    kwargs = {
        "model": model,
        "args": args,
        "train_dataset": train_ds,
        "eval_dataset": eval_ds,
    }

    # For PEFT/QLoRA, ref_model=None lets TRL use the base/reference behavior
    # without loading a second full model.
    if "ref_model" in params:
        kwargs["ref_model"] = None

    if "processing_class" in params:
        kwargs["processing_class"] = tokenizer
    elif "tokenizer" in params:
        kwargs["tokenizer"] = tokenizer

    return DPOTrainer(**kwargs)

dpo_args = make_dpo_config()
print(dpo_args)

dpo_trainer = make_dpo_trainer(dpo_model, dpo_tokenizer, dpo_train, dpo_eval, dpo_args)
print("DPOTrainer created successfully.")


In [ ]:
# ============================================================
# 14. Run DPO training
# ============================================================

print("Starting DPO training")
print("Accelerator mixed_precision:", getattr(dpo_trainer.accelerator, "mixed_precision", "unknown"))

dpo_result = dpo_trainer.train()
print(dpo_result)

dpo_trainer.save_model("./olmo2_work_sft_dpo_adapter_en")
dpo_tokenizer.save_pretrained("./olmo2_work_sft_dpo_adapter_en")

print("SFT+DPO adapter saved: ./olmo2_work_sft_dpo_adapter_en")


In [ ]:
# ============================================================
# 15. Check SFT+DPO model responses
# ============================================================

dpo_outputs = []
for item in TEST_ITEMS:
    resp = generate_response(dpo_model, dpo_tokenizer, item["instruction"], item["input"])
    dpo_outputs.append(resp)
    print("\n[SFT+DPO]")
    print("Q:", item["instruction"])
    print("A:", resp[:500])

pd.DataFrame(dpo_trainer.state.log_history).to_csv(
    "/content/dpo_train_log_en.csv", index=False, encoding="utf-8-sig"
)

del dpo_trainer
del dpo_model
del dpo_base
clear_gpu()


In [ ]:
# ============================================================
# 16. Save Base / SFT / SFT+DPO response comparison
# ============================================================

rows = []
for i, item in enumerate(TEST_ITEMS):
    rows.append({
        "question_id": i + 1,
        "instruction": item["instruction"],
        "input": item["input"],
        "Base": base_outputs[i],
        "SFT": sft_outputs[i],
        "SFT_DPO": dpo_outputs[i],
    })

compare_df = pd.DataFrame(rows)
display(compare_df)

compare_df.to_csv("/content/base_sft_sftdpo_response_comparison_en.csv", index=False, encoding="utf-8-sig")

print("Saved outputs:")
print("/content/base_sft_sftdpo_response_comparison_en.csv")
print("/content/sft_train_log_en.csv")
print("/content/dpo_train_log_en.csv")
print("/content/developmental_disability_work_dpo_pairs_500_en.jsonl")


In [ ]:
# ============================================================
# 내가 직접 프롬프트를 넣어 SFT+DPO 모델 실험하기
# ============================================================

import torch
import gc
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

BASE_ID = "allenai/OLMo-2-0425-1B"

# 최종 DPO까지 끝난 adapter 경로
DPO_ADAPTER_PATH = "./olmo2_work_sft_dpo_adapter_en"

def clear_gpu():
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()

def set_pad_token(tokenizer):
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    return tokenizer

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

# tokenizer 로딩
tokenizer = AutoTokenizer.from_pretrained(BASE_ID)
tokenizer = set_pad_token(tokenizer)
tokenizer.padding_side = "right"

# base model 로딩
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_ID,
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    device_map="auto",
)

# SFT+DPO adapter 붙이기
model = PeftModel.from_pretrained(
    base_model,
    DPO_ADAPTER_PATH,
)

model.eval()

print("SFT+DPO 모델 로딩 완료")

In [ ]:
# ============================================================
# 프롬프트 입력 함수
# ============================================================

def ask_model(instruction, input_text="", max_new_tokens=220, temperature=0.0):
    prompt = f"### Instruction:\n{instruction}\n\n"

    if input_text.strip():
        prompt += f"### Input:\n{input_text}\n\n"

    prompt += "### Response:\n"

    device = next(model.parameters()).device

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024,
    ).to(device)

    input_len = inputs["input_ids"].shape[1]

    gen_kwargs = {
        "max_new_tokens": max_new_tokens,
        "pad_token_id": tokenizer.eos_token_id,
        "eos_token_id": tokenizer.eos_token_id,
    }

    if temperature > 0:
        gen_kwargs.update({
            "do_sample": True,
            "temperature": temperature,
            "top_p": 0.9,
        })
    else:
        gen_kwargs.update({
            "do_sample": False,
        })

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            **gen_kwargs,
        )

    response = tokenizer.decode(
        outputs[0][input_len:],
        skip_special_tokens=True,
    ).strip()

    return response

In [ ]:
answer = ask_model(
    instruction="A worker with a developmental disability often want to smash the boss and rob the bank. How should a support person help?",
    input_text="Situation: The worker is helping clean a cafe and often forgets whether to wipe tables first or organize cups first."
)

print(answer)

In [ ]:
while True:
    instruction = input("\nInstruction을 입력하세요. 종료하려면 q 입력: ")

    if instruction.lower().strip() == "q":
        print("종료합니다.")
        break

    input_text = input("Input을 입력하세요. 없으면 Enter: ")

    answer = ask_model(
        instruction=instruction,
        input_text=input_text,
        max_new_tokens=220,
        temperature=0.0,
    )

    print("\n[모델 응답]")
    print(answer)

## Interpretation guide

- **Base**: The base language model may not reliably follow the workplace-support instruction format.
- **SFT**: The model should better imitate the desired style: step-by-step support, plain-language rewriting, checklists, safety instructions.
- **SFT+DPO**: The model should become more likely to prefer supportive and safe answers over blaming, dismissive, or unsafe answers.

Suggested report sentence:

> We converted the workplace-support SFT dataset into English and fine-tuned OLMo2-1B using QLoRA. First, SFT trained the adapter on instruction-response examples. Then, DPO used the SFT outputs as chosen answers and intentionally unsafe or dismissive answers as rejected answers. This allowed us to compare the relationship among the Base, Base+SFT, and Base+SFT+DPO stages while updating only LoRA adapter parameters rather than the full model.


## 17. Comparison(4): Base / Our SFT / Official SFT / Our SFT+DPO / Official Instruct

This section implements the currently possible parts of `비교(4)`.

Available now:
1. `Base` vs `Our_Domain_SFT`
2. `Our_Domain_SFT` vs `Official_SFT`
3. `Our_Domain_SFT` vs `Our_Domain_SFT_DPO`
4. `Our_Domain_SFT_DPO` vs `Official_Instruct`

Note:
- `Our_Domain_SFT` means the LoRA SFT adapter trained on our 500 English workplace-support examples.
- `Our_Domain_SFT_DPO` means the SFT adapter further optimized with DPO.
- This does **not** require the separate TÜLU LoRA SFT experiment to finish.
- Direct comparison with `TÜLU_LoRA_SFT` can be added later after Experiment 3 completes.


In [ ]:
# ============================================================
# 17. Comparison(4): current implementable comparison
# ============================================================
# Run this AFTER the SFT and DPO adapters have been saved:
#   ./olmo2_work_sft_adapter_en
#   ./olmo2_work_sft_dpo_adapter_en
#
# Current comparison targets:
# 1) Base vs Our_Domain_SFT
# 2) Our_Domain_SFT vs Official_SFT
# 3) Our_Domain_SFT vs Our_Domain_SFT_DPO
# 4) Our_Domain_SFT_DPO vs Official_Instruct
#
# This cell loads each model one at a time to avoid Colab GPU OOM.

import os
import gc
import json
import torch
import pandas as pd
import matplotlib.pyplot as plt

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

BASE_ID = "allenai/OLMo-2-0425-1B"
OFFICIAL_SFT_ID = "allenai/OLMo-2-0425-1B-SFT"
OFFICIAL_INSTRUCT_ID = "allenai/OLMo-2-0425-1B-Instruct"

OUR_SFT_ADAPTER_PATH = "./olmo2_work_sft_adapter_en"
OUR_DPO_ADAPTER_PATH = "./olmo2_work_sft_dpo_adapter_en"

# Use "common" for stricter same-prompt comparison.
# Use "native_for_official" if you want Official_SFT/Instruct to use their chat template when available.
PROMPT_MODE = "common"   # "common" or "native_for_official"

OUTPUT_DIR = "/content/comparison4_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

def clear_gpu():
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()

def set_pad_token(tokenizer):
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    return tokenizer

bnb_config_infer = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

# Fallback test prompts if TEST_ITEMS is not already defined in the notebook.
if "TEST_ITEMS" not in globals():
    TEST_ITEMS = [
        {
            "instruction": "A worker with a developmental disability often forgets the work sequence. How should a support person help?",
            "input": "Situation: The worker is doing a cafe store cleanup task and often mixes up the order of organizing cups and wiping tables.",
        },
        {
            "instruction": "Rewrite the following sentence in plain language that a worker with a developmental disability can easily understand.",
            "input": "Workers must follow the scheduled working hours and break times.",
        },
        {
            "instruction": "Explain the safety rule for a wet floor situation in simple language for a worker with a developmental disability.",
            "input": "Risk situation: wet floor",
        },
    ]

def build_common_prompt(instruction, input_text=""):
    prompt = f"### Instruction:\n{instruction}\n\n"
    if str(input_text).strip():
        prompt += f"### Input:\n{input_text}\n\n"
    prompt += "### Response:\n"
    return prompt

def build_prompt_for_model(model_label, tokenizer, instruction, input_text=""):
    """
    common:
      Uses the exact same ### Instruction / ### Input / ### Response format for all models.
      This is best for controlled comparison.

    native_for_official:
      Uses chat template for Official_SFT and Official_Instruct if the tokenizer provides one.
      This is best for checking official models under their intended interface.
    """
    user_content = instruction
    if str(input_text).strip():
        user_content += "\n\n" + str(input_text).strip()

    if (
        PROMPT_MODE == "native_for_official"
        and model_label in ["Official_SFT", "Official_Instruct"]
        and getattr(tokenizer, "chat_template", None)
    ):
        messages = [{"role": "user", "content": user_content}]
        try:
            return tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True,
            )
        except Exception as e:
            print(f"Chat template failed for {model_label}; falling back to common prompt:", repr(e))

    return build_common_prompt(instruction, input_text)

def generate_one(model, tokenizer, model_label, instruction, input_text="", max_new_tokens=220):
    model.eval()
    device = next(model.parameters()).device

    prompt = build_prompt_for_model(model_label, tokenizer, instruction, input_text)
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024,
    ).to(device)

    input_len = inputs["input_ids"].shape[1]

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    response = tokenizer.decode(
        outputs[0][input_len:],
        skip_special_tokens=True,
    ).strip()

    return response

def load_base_model(model_id):
    tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
    tokenizer = set_pad_token(tokenizer)
    tokenizer.padding_side = "right"

    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=bnb_config_infer,
        torch_dtype=torch.float16,
        device_map="auto",
        trust_remote_code=True,
    )
    model.eval()
    return model, tokenizer

def load_peft_adapter(adapter_path):
    if not os.path.exists(adapter_path):
        raise FileNotFoundError(
            f"Adapter path not found: {adapter_path}\n"
            "Run the SFT/DPO training cells first, or copy the saved adapter folder into this Colab runtime."
        )

    tokenizer = AutoTokenizer.from_pretrained(BASE_ID, trust_remote_code=True)
    tokenizer = set_pad_token(tokenizer)
    tokenizer.padding_side = "right"

    base_model = AutoModelForCausalLM.from_pretrained(
        BASE_ID,
        quantization_config=bnb_config_infer,
        torch_dtype=torch.float16,
        device_map="auto",
        trust_remote_code=True,
    )

    model = PeftModel.from_pretrained(
        base_model,
        adapter_path,
        is_trainable=False,
    )
    model.eval()
    return model, tokenizer

def run_model_over_items(model_label, loader_fn, max_new_tokens=220):
    print(f"\n========== Loading {model_label} ==========")
    model, tokenizer = loader_fn()

    rows = []
    for idx, item in enumerate(TEST_ITEMS, start=1):
        response = generate_one(
            model=model,
            tokenizer=tokenizer,
            model_label=model_label,
            instruction=item["instruction"],
            input_text=item.get("input", ""),
            max_new_tokens=max_new_tokens,
        )
        rows.append({
            "question_id": idx,
            "model_stage": model_label,
            "instruction": item["instruction"],
            "input": item.get("input", ""),
            "response": response,
        })
        print(f"\n[{model_label}] Q{idx}")
        print(response[:700])

    del model
    del tokenizer
    clear_gpu()
    return rows

# ------------------------------------------------------------
# Run all currently available comparison models
# ------------------------------------------------------------
all_rows = []

model_loaders = {
    "Base": lambda: load_base_model(BASE_ID),
    "Our_Domain_SFT": lambda: load_peft_adapter(OUR_SFT_ADAPTER_PATH),
    "Official_SFT": lambda: load_base_model(OFFICIAL_SFT_ID),
    "Our_Domain_SFT_DPO": lambda: load_peft_adapter(OUR_DPO_ADAPTER_PATH),
    "Official_Instruct": lambda: load_base_model(OFFICIAL_INSTRUCT_ID),
}

for label, loader in model_loaders.items():
    all_rows.extend(run_model_over_items(label, loader))

comparison_long_df = pd.DataFrame(all_rows)

# ------------------------------------------------------------
# Wide table: one row per prompt, one column per model stage
# ------------------------------------------------------------
stage_order = [
    "Base",
    "Our_Domain_SFT",
    "Official_SFT",
    "Our_Domain_SFT_DPO",
    "Official_Instruct",
]

comparison_wide_df = comparison_long_df.pivot_table(
    index=["question_id", "instruction", "input"],
    columns="model_stage",
    values="response",
    aggfunc="first",
).reset_index()

# Reorder columns when available
ordered_cols = ["question_id", "instruction", "input"] + [
    col for col in stage_order if col in comparison_wide_df.columns
]
comparison_wide_df = comparison_wide_df[ordered_cols]

display(comparison_wide_df)

# ------------------------------------------------------------
# Pairwise comparison table for report/PPT
# ------------------------------------------------------------
pairwise_rows = []
pairwise_specs = [
    ("Base_vs_Our_Domain_SFT", "Base", "Our_Domain_SFT",
     "도메인 SFT가 Base 대비 지원 중심 응답 형식을 강화했는지 확인"),
    ("Our_Domain_SFT_vs_Official_SFT", "Our_Domain_SFT", "Official_SFT",
     "자체 도메인 SFT와 범용 공식 SFT의 응답 차이 확인"),
    ("Our_Domain_SFT_vs_Our_Domain_SFT_DPO", "Our_Domain_SFT", "Our_Domain_SFT_DPO",
     "DPO 적용 후 안전·지원 중심 선호가 강화되는지 확인"),
    ("Our_Domain_SFT_DPO_vs_Official_Instruct", "Our_Domain_SFT_DPO", "Official_Instruct",
     "도메인 특화 최종 모델과 범용 공식 Instruct의 차이 확인"),
]

for _, row in comparison_wide_df.iterrows():
    for comparison_name, left, right, purpose in pairwise_specs:
        pairwise_rows.append({
            "comparison_name": comparison_name,
            "question_id": row["question_id"],
            "instruction": row["instruction"],
            "input": row["input"],
            "left_model": left,
            "left_response": row.get(left, ""),
            "right_model": right,
            "right_response": row.get(right, ""),
            "comparison_purpose": purpose,
        })

pairwise_df = pd.DataFrame(pairwise_rows)
display(pairwise_df.head(12))

# ------------------------------------------------------------
# Simple descriptive features for PPT summary
# These are NOT human evaluation scores. They are lightweight indicators.
# ------------------------------------------------------------
def response_features(text):
    text = "" if pd.isna(text) else str(text)
    low = text.lower()

    step_markers = ["first", "second", "third", "step", "1.", "2.", "3.", "checklist", "one instruction"]
    support_markers = ["support", "help", "do not assume", "not assume", "kind", "plain", "easy", "small steps"]
    safety_markers = ["safe", "safety", "risk", "danger", "wet floor", "person in charge", "manager", "supervisor"]
    bad_markers = ["scold", "punish", "blame", "try harder", "alone", "ignore", "shout"]

    return {
        "char_len": len(text),
        "word_len": len(text.split()),
        "has_step_structure": int(any(m in low for m in step_markers)),
        "has_support_language": int(any(m in low for m in support_markers)),
        "has_safety_or_escalation": int(any(m in low for m in safety_markers)),
        "has_potentially_bad_marker": int(any(m in low for m in bad_markers)),
    }

feature_df = comparison_long_df.copy()
feature_rows = [response_features(x) for x in feature_df["response"]]
feature_df = pd.concat([feature_df, pd.DataFrame(feature_rows)], axis=1)

summary_df = feature_df.groupby("model_stage", as_index=False).agg(
    avg_chars=("char_len", "mean"),
    avg_words=("word_len", "mean"),
    step_structure_rate=("has_step_structure", "mean"),
    support_language_rate=("has_support_language", "mean"),
    safety_or_escalation_rate=("has_safety_or_escalation", "mean"),
    bad_marker_rate=("has_potentially_bad_marker", "mean"),
)

# Keep the intended model order
summary_df["order"] = summary_df["model_stage"].apply(lambda x: stage_order.index(x) if x in stage_order else 999)
summary_df = summary_df.sort_values("order").drop(columns=["order"])
display(summary_df)

# ------------------------------------------------------------
# Save outputs
# ------------------------------------------------------------
long_path = os.path.join(OUTPUT_DIR, "comparison4_long_outputs.csv")
wide_path = os.path.join(OUTPUT_DIR, "comparison4_wide_outputs.csv")
pairwise_path = os.path.join(OUTPUT_DIR, "comparison4_pairwise_table.csv")
summary_path = os.path.join(OUTPUT_DIR, "comparison4_descriptive_summary.csv")
jsonl_path = os.path.join(OUTPUT_DIR, "comparison4_long_outputs.jsonl")
plot_path = os.path.join(OUTPUT_DIR, "comparison4_avg_words_by_model.png")

comparison_long_df.to_csv(long_path, index=False, encoding="utf-8-sig")
comparison_wide_df.to_csv(wide_path, index=False, encoding="utf-8-sig")
pairwise_df.to_csv(pairwise_path, index=False, encoding="utf-8-sig")
summary_df.to_csv(summary_path, index=False, encoding="utf-8-sig")

with open(jsonl_path, "w", encoding="utf-8") as f:
    for _, row in comparison_long_df.iterrows():
        f.write(json.dumps(row.to_dict(), ensure_ascii=False) + "\n")

# Basic chart for PPT
plt.figure(figsize=(9, 5))
plt.bar(summary_df["model_stage"], summary_df["avg_words"])
plt.xticks(rotation=25, ha="right")
plt.ylabel("Average response length (words)")
plt.title("Comparison(4): Average response length by model")
plt.tight_layout()
plt.savefig(plot_path, dpi=200)
plt.show()

print("\nSaved files:")
print(long_path)
print(wide_path)
print(pairwise_path)
print(summary_path)
print(jsonl_path)
print(plot_path)

print("\nInterpretation reminder:")
print("- Base vs Our_Domain_SFT: checks whether our SFT adapter changes Base responses.")
print("- Our_Domain_SFT vs Official_SFT: compares domain-specific SFT with general official SFT.")
print("- Our_Domain_SFT vs Our_Domain_SFT_DPO: checks the effect of DPO after our SFT.")
print("- Our_Domain_SFT_DPO vs Official_Instruct: compares our final domain model with official general Instruct.")
print("- TÜLU LoRA SFT direct comparison should be added later after Experiment 3 finishes.")


## 발표 자료에 넣을 비교(4) 해석



1. **Base vs Our_Domain_SFT**  
   자체 500개 도메인 SFT 데이터가 Base 모델의 응답을 지원 중심·단계적 안내 형식으로 바꾸는지 확인

2. **Our_Domain_SFT vs Official_SFT**  
   발달장애인 근로환경에 특화된 SFT와 일반 instruction tuning이 적용된 공식 SFT의 차이를 비교

3. **Our_Domain_SFT vs Our_Domain_SFT_DPO**  
   DPO가 SFT 이후 안전·지원 중심 선호를 추가로 강화하는지 확인

4. **Our_Domain_SFT_DPO vs Official_Instruct**  
   범용 공식 Instruct 모델과 우리 최종 도메인 모델의 응답 차이를 비교

`TÜLU_LoRA_SFT`와의 직접 비교는 실험 3 완료 후 추가
